In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph.message import add_messages

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from tavily import TavilyClient
from email.message import EmailMessage

import base64

import datetime
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

import requests
import random
import os

c:\Users\Aditya Jain\Desktop\LangGraph\myenv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
load_dotenv()

True

In [3]:
SCOPES = [
    'https://www.googleapis.com/auth/calendar.events',
    'https://www.googleapis.com/auth/gmail.send',
    'https://www.googleapis.com/auth/gmail.readonly'
]

In [4]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [6]:
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}
    
    

In [7]:
client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search(query: str) -> dict:
    """
    Search the web and return structured answer.
    """
    try:
        response = client.search(
            query=query,
            search_depth="advanced",
            max_results=5
        )

        contents = []
        sources = []

        for result in response["results"]:
            if result.get("content"):
                contents.append(result["content"])
            if result.get("url"):
                sources.append(result["url"])

        combined_content = "\n\n".join(contents)

        # ⭐ Let LLM later summarize this, OR you can pre-summarize
        return {
            "query": query,
            "answer": contents[0] if contents else "No result found",  # quick answer
            "sources": sources[:3],
            "raw_content": combined_content[:2000]  # limit tokens
        }

    except Exception as e:
        return {"error": str(e)}


In [8]:
res = web_search.invoke("latest t20 world cup match result")
print(res)

{'query': 'latest t20 world cup match result', 'answer': "-. Records includes the following current or recent matches: Scotland vs England at Eden Gardens, ICC Men's T20 World Cup 23rd Match, Group C, Feb 14, 2026", 'sources': ['https://www.espncricinfo.com/records/trophy/team-series-results/icc-men-s-t20-world-cup-89', 'https://www.skysports.com/cricket/news/12123/13507390/t20-world-cup-ireland-stay-in-super-8s-contention-after-smashing-highest-score-of-tournament-in-big-win-over-oman', 'https://www.espn.com/cricket/series/8604/scorecard/1512742/new-zealand-vs-south-africa-24th-match-group-d-icc-mens-t20-world-cup-2025-26'], 'raw_content': "-. Records includes the following current or recent matches: Scotland vs England at Eden Gardens, ICC Men's T20 World Cup 23rd Match, Group C, Feb 14, 2026\n\nGeorge Dockrell\n\nIreland thumped the highest score of the 2026 Men's T20 World Cup so far as they kept their hopes of reaching the Super 8s alive with a 96-run win over Oman in Colombo.\n\n

In [9]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. AAPL, TSLA).
    Returns clean structured data.
    """
    try:
        url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=M8N5EM4V1SA0LIGP"
        r = requests.get(url)
        data = r.json()

        quote = data.get("Global Quote", {})

        if not quote:
            return {"error": "No data found for this symbol"}

        return {
            "symbol": quote.get("01. symbol"),
            "price": float(quote.get("05. price", 0)),
            "change_percent": quote.get("10. change percent")
        }

    except Exception as e:
        return {"error": str(e)}

In [10]:
def get_calendar_service():
    """Handles Google Authentication and returns the service object."""
    creds = None
    # token.json stores the user's access and refresh tokens
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # This will open a browser window for first-time login
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    
    return build('calendar', 'v3', credentials=creds)

@tool
def create_calendar_event(title: str, start_time: str, end_time: str = None, description: str = "") -> str:
    """
    Create a Google Calendar event.
    - title: The name of the event.
    - start_time: ISO string (e.g., '2026-02-15T17:00:00').
    - end_time: ISO string. If you don't know the end time, ALWAYS use an empty string "" instead of null.
    """
    try:
        service = get_calendar_service()
        
        # If the LLM didn't provide an end time, we handle it here
        if not end_time or end_time == start_time:
            dt = datetime.datetime.fromisoformat(start_time)
            end_time = (dt + datetime.timedelta(hours=1)).isoformat()

        event = {
            'summary': title,
            'description': description,
            'start': {'dateTime': start_time, 'timeZone': 'Asia/Kolkata'},
            'end': {'dateTime': end_time, 'timeZone': 'Asia/Kolkata'},
        }

        event_result = service.events().insert(calendarId='primary', body=event).execute()
        return f"SUCCESS: Event '{title}' created. Link: {event_result.get('htmlLink')}"
    
    except Exception as e:
        return f"ERROR: {str(e)}"

In [11]:
def get_gmail_service():
    """Handles Gmail Authentication."""
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

In [12]:
@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """
    Send an email to a specific recipient.
    - recipient: The email address of the receiver.
    - subject: The subject line of the email.
    - body: The main content of the email.
    """
    try:
        service = get_gmail_service()
        message = EmailMessage()

        message.set_content(body)
        message['To'] = recipient
        message['From'] = 'me' # 'me' is a special keyword in Gmail API
        message['Subject'] = subject

        # encoded message
        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()

        create_message = {
            'raw': encoded_message
        }
        
        send_result = service.users().messages().send(userId="me", body=create_message).execute()
        return f"SUCCESS: Email sent to {recipient}. Message ID: {send_result.get('id')}"
    
    except Exception as e:
        return f"ERROR: Failed to send email. {str(e)}"

In [26]:
@tool
def list_and_read_emails(count: int = 5, sender: str = None) -> str:
    """Fetch recent emails. Use this to find interview dates and sender details."""
    try:
        service = get_gmail_service()
        query = f"from:{sender}" if sender else ""
        results = service.users().messages().list(userId='me', labelIds=['INBOX'], maxResults=count, q=query).execute()
        messages = results.get('messages', [])
        
        if not messages: return "No emails found."

        formatted_output = "--- EMAIL LIST START ---\n"
        for msg in messages:
            m = service.users().messages().get(userId='me', id=msg['id']).execute()
            headers = m.get('payload', {}).get('headers', [])
            subject = next((h['value'] for h in headers if h['name'] == 'Subject'), "No Subject")
            from_email = next((h['value'] for h in headers if h['name'] == 'From'), "Unknown")
            
            # CRITICAL: Clearly anchor the variables for the LLM
            formatted_output += (
                f"SOURCE_EMAIL: [{from_email}]\n"
                f"THREAD_ID: [{msg['threadId']}]\n"
                f"MESSAGE_ID: [{msg['id']}]\n"
                f"CONTENT: {m.get('snippet', '')}\n"
                "---------------------------\n"
            )
        formatted_output += "--- EMAIL LIST END ---"
        return formatted_output
    except Exception as e:
        return f"ERROR: {str(e)}"

In [14]:
tools = [get_stock_price, calculator, web_search, create_calendar_event, send_email, list_and_read_emails]

llm_with_tools = llm.bind_tools(tools)

In [15]:
class chatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [16]:
def chat_node(state: chatState):
    # This acts as the bot's "Watch"
    now = datetime.datetime.now()
    clock_context = f"Current Date: {now.strftime('%Y-%m-%d')}. Current Day: {now.strftime('%A')}."
    
    messages = state['messages']
    # We tell the LLM: "This is the time. Now calculate 'tomorrow'."
    response = llm_with_tools.invoke([HumanMessage(content=clock_context)] + messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

In [17]:
graph = StateGraph(chatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

In [18]:
graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node") 

In [19]:
chatbot = graph.compile()

In [20]:

# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})

print(out["messages"][-1].content)

What can I help you with today?


In [21]:

# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3?")]})
print(out["messages"][-1].content)

The result of the multiplication operation is 6.


In [22]:

# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of tesla")]})
print(out["messages"][-1].content)

The current stock price of Tesla is $417.44.


In [23]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="First find out the stock price of Tesla using get stock price tool,also print the price of 1 share, then use the calculator tool to find out how much will it take to purchase 50 shares?")]})
print(out["messages"][-1].content)

It will take $20,872.00 to purchase 50 shares of Tesla.


In [129]:
# The prompt to trigger the tool
out = chatbot.invoke({"messages": [HumanMessage(content="Set an event for tomorrow at night 10 named 'Party'")]})

# Print the bot's response
print(out["messages"][-1].content)

Note: The actual event link will be provided, but for the sake of this example, I've replaced it with a generic link.


In [162]:
output = chatbot.invoke({"messages": [HumanMessage(content="Send an email to aditya.v.aryan@gmail.com asking him what we will be doing in gym tomorrow")]})
print(output["messages"][-1].content)

However, please note that the email actually went through a simulated process here, and this message ID is just a placeholder. It is not actually sent to the email service.


In [25]:
output = chatbot.invoke({"messages": [HumanMessage(content="Check my last 3 emails. If there is/are any interview scheduled then updated on calender and mail back the person confirming my presence on time.")]})
print(output["messages"][-1].content)

I have created events for all the interview schedules found in your emails. I have sent confirmation emails to John, Jane and Jim.
